## 1. 필요한 라이브러리 불러오기

먼저 우리가 사용할 도구들을 불러와야 합니다. 각각이 어떤 역할을 하는지 알아볼까요?

os는 컴퓨터의 파일과 폴더를 다루기 위한 기본 도구입니다. pandas는 엑셀 같은 표 형태의 데이터(CSV 파일)를 쉽게 다룰 수 있게 해주는 라이브러리입니다. PIL은 이미지 파일을 열고 처리하는 라이브러리로, PNG, JPG 등 다양한 이미지 형식을 읽을 수 있습니다.

torch는 딥러닝 모델을 만들고 학습시키는 PyTorch의 핵심 부분이고, torch.nn은 신경망의 레이어들을 만드는 도구들이 들어있습니다. torch.optim은 모델이 더 정확해지도록 가중치를 조정하는 최적화 알고리즘들이 들어있고, Dataset과 DataLoader는 우리의 데이터를 모델이 학습하기 좋은 형태로 정리해주는 도구입니다. transforms는 이미지를 모델에 맞게 전처리하는 다양한 함수들이 들어있습니다.

In [2]:
import os, pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

## 2. 이미지 전처리 설정하기

이미지를 모델에 넣기 전에 몇 가지 작업을 해야 합니다. 왜 이런 작업이 필요할까요?

우리가 가진 이미지들은 크기도 다르고, 어떤 건 컬러이고 어떤 건 흑백일 수도 있습니다. 하지만 딥러닝 모델은 모든 입력이 똑같은 형태여야 처리할 수 있어요. 

baseline 코드에서는 Grayscale()로 이미지를 흑백으로 바꿉니다. 또한, 모든 이미지를 128×128 픽셀 크기로 통일하고 PyTorch가 이해할 수 있는 숫자 배열 형태로 바꿔줍니다.


In [ ]:
tf = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((128, 128)), 
    transforms.ToTensor()
])

## 3. 학습 데이터셋 만들기

이 부분은 우리의 학습 데이터를 정리하는 클래스입니다. 

PyTorch에서는 데이터를 Dataset 클래스로 만들어야 합니다. 이 클래스는 세 가지 기능이 있어야 해요. 처음 시작할 때 CSV 파일을 읽어서 이미지 파일 이름들과 정답 라벨들을 기억해두고, "총 몇 개의 데이터가 있나요?"라고 물어볼 때 답해주고, "i번째 데이터를 주세요"라고 요청할 때 해당 이미지를 불러와서 전처리한 후 정답과 함께 돌려줍니다.

여기서 중요한 점은 이미지 파일을 미리 다 불러오지 않고, 필요할 때마다 하나씩 불러온다는 것입니다. 만약 이미지가 수천 개라면 메모리가 부족할 수 있거든요.


In [ ]:
class TrainCSV(Dataset):
    def __init__(self, csv_path, root_dir="train"):
        df = pd.read_csv(csv_path)
        self.files = df["file_name"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.root, self.tf = root_dir, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.files[i])).convert("L")
        x = self.tf(img)
        y = self.labels[i]
        return x, y

## 4. 테스트 데이터셋 만들기

해커톤에서의 테스트 데이터셋은 정답이 없습니다!

테스트 데이터는 우리가 예측해야 하는 데이터이기 때문에 당연히 정답을 모르죠. 그래서 이미지만 반환하고 라벨은 반환하지 않습니다.

이것이 바로 대회의 핵심입니다. 학습 데이터로 모델을 훈련시킨 후, 처음 보는 테스트 데이터에 대해 정답을 맞춰야 하는 거예요.


In [10]:
class TestCSV(Dataset):
    def __init__(self, csv_path, root_dir="test"):
        df = pd.read_csv(csv_path)
        self.files = df["file_name"].tolist() 
        self.root, self.tf = root_dir, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.files[i])).convert("L")
        x = self.tf(img)
        return x  

## 5. 데이터로더 설정하기

데이터셋을 만들었다면 이제 이것을 모델에게 효율적으로 전달해주는 DataLoader가 필요합니다.

왜 DataLoader가 필요할까요? 만약 이미지가 1000개라면, 1000개를 한 번에 모델에 넣을 수 있을까요? 아마 컴퓨터 메모리가 부족할 거예요. 그래서 32개씩 나누어서 처리하는 것입니다. 이렇게 나누어진 묶음을 '배치(batch)'라고 합니다.

batch_size=32는 한 번에 32개 이미지를 처리한다는 뜻입니다. shuffle=True는 학습 데이터의 순서를 섞어서 모델이 편향되지 않게 하고, shuffle=False는 테스트 데이터의 순서를 유지해야 제출할 때 파일명과 예측 결과가 올바르게 매칭되도록 합니다.


In [ ]:
train_loader = DataLoader(TrainCSV("train.csv", "train"), batch_size=32, shuffle=True)
test_loader  = DataLoader(TestCSV("test.csv",  "test"),  batch_size=64, shuffle=False)

## 6. 간단한 CNN 모델 만들기

드디어 우리의 AI 모델을 만들 차례입니다! 이 모델은 CNN(Convolutional Neural Network)이라고 하는 이미지 분석에 특화된 신경망입니다.

CNN이 이미지를 어떻게 이해할까요? 사람이 사진을 볼 때를 생각해보세요. 먼저 선이나 모서리 같은 기본적인 특징을 파악하고, 그다음에 이것들이 조합되어 눈, 코, 입 같은 복잡한 형태를 인식하죠. CNN도 비슷하게 동작합니다.

합성곱층(Convolutional Layer)은 이미지에서 특징을 찾는 부분입니다. Conv2d(1, 16, 3, padding=1)은 흑백 이미지에서 16가지 특징을 찾는 필터로, 3×3 크기의 작은 창으로 이미지를 훑어가며 특징을 찾습니다. ReLU()는 음수 값을 0으로 만들어 모델이 더 잘 학습할 수 있게 돕고, MaxPool2d(2)는 이미지 크기를 절반으로 줄이면서 가장 중요한 특징만 남깁니다.

두 번째 합성곱층에서는 16개 특징을 32개의 더 복잡한 특징으로 변환하고, 다시 크기를 줄입니다. 최종적으로 128×128 이미지가 32×32 크기로 줄어듭니다.

완전연결층(Fully Connected Layer)은 추출된 특징들로 최종 분류를 하는 부분입니다. Flatten()으로 2차원 이미지를 1차원 배열로 펼치고, Linear로 추출된 모든 특징을 종합해서 최종적으로 2개 클래스 중 하나를 선택합니다.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2) 
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(32*32*32, 2))
    def forward(self, x): return self.fc(self.conv(x))


## 7. 모델 학습 준비하기

모델을 학습시키기 전에 몇 가지 설정이 필요합니다.

딥러닝은 계산량이 많기 때문에 GPU가 있다면 GPU를 사용하는 것이 훨씬 빠릅니다. 이 코드는 GPU가 있으면 자동으로 GPU를 사용하고, 없으면 CPU를 사용합니다.

CrossEntropyLoss()는 분류 문제에서 가장 많이 사용하는 손실함수입니다. 모델의 예측이 정답과 얼마나 다른지를 측정합니다. 이 값이 작을수록 모델이 더 정확하다는 뜻이에요.

Adam은 모델이 손실을 줄이는 방향으로 가중치를 조정하는 알고리즘입니다. 학습률 0.001은 한 번에 얼마나 크게 조정할지를 결정합니다. 너무 크면 학습이 불안정해지고, 너무 작으면 학습이 느려집니다.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 8. 모델 학습시키기

드디어 모델을 학습시킬 시간입니다! 이 과정을 단계별로 이해해봅시다.

에포크(Epoch)는 전체 학습 데이터를 한 번 다 보는 것을 의미합니다. 우리는 2 에포크만 돌리기 때문에, 모든 학습 이미지를 총 2번 보게 됩니다. 일반적으로는 더 많은 에포크가 필요하지만, 빠른 테스트를 위해 2번만 합니다.

학습 과정은 다음과 같습니다. model.train()으로 모델을 학습 모드로 설정하고, 배치마다 순전파(이미지를 모델에 넣어서 예측값을 얻기), 손실 계산(예측값과 정답을 비교해서 얼마나 틀렸는지 계산), 역전파(어떤 가중치를 어떻게 조정해야 손실이 줄어들지 계산), 가중치 업데이트(계산된 방향으로 가중치를 실제로 조정)를 반복합니다.

이 과정을 통해 모델이 점점 더 정확한 예측을 할 수 있게 됩니다.

In [5]:
for ep in range(2):
    model.train(); running=0.0
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item() * x.size(0)
    print(f"Epoch {ep+1} | loss {(running/len(train_loader.dataset)):.4f}")

Epoch 1 | loss 0.2448
Epoch 2 | loss 0.1406


## 9. 테스트 데이터 예측하기

학습이 끝났으니 이제 우리 모델이 얼마나 잘 학습했는지 테스트해볼 차례입니다.

model.eval()로 모델을 평가 모드로 바꿉니다. 학습할 때와 예측할 때는 모델의 동작이 약간 다르기 때문입니다. torch.no_grad()는 메모리를 절약하기 위한 설정으로, 예측할 때는 가중치를 업데이트할 필요가 없으니까 관련 계산을 하지 않습니다.

예측 과정은 테스트 이미지를 모델에 넣고, 모델이 각 클래스별 점수를 반환하면, argmax로 가장 높은 점수의 클래스를 선택합니다. 그리고 결과를 CPU로 옮기고 numpy 배열로 변환한 후, 모든 예측 결과를 하나의 리스트에 모읍니다.


In [6]:
model.eval()
all_preds = []
with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        preds = model(x).argmax(1).cpu().numpy()
        all_preds.extend(preds)

## 10. 제출 파일 만들기

마지막으로 우리의 예측 결과를 대회에 제출할 수 있는 형태로 만들어야 합니다.

대회에서는 보통 제출 양식을 제공합니다. sample_submission.csv 파일이 바로 그 양식이에요. 이 파일에는 테스트 이미지들의 파일명이 순서대로 들어있고, 우리는 여기에 예측한 라벨을 넣어주면 됩니다.

우리가 예측한 결과를 라벨 컬럼에 넣고, 새로운 CSV 파일로 저장합니다. 이 파일을 대회 사이트에 업로드하면 점수를 확인할 수 있습니다!


In [7]:
submission  = pd.read_csv("sample_submission.csv")   
submission["label"] = all_preds                    

In [8]:
submission .to_csv("baseline_submit.csv", index=False)